In [11]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn xgboost sweetviz

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.0 MB 19.7 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 27.6 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:


import pandas as pd
import numpy as np
import sweetviz as sv

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

In [13]:
file_path = r"C:\Users\2003n\Downloads\creditcard.csv"
df = pd.read_csv(file_path)

print(df.head())


   LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0     400000    1          1         2   32      0      0      0      0   
1     120000    2          2         2   30     -1     -1     -1     -1   
2     270000    2          2         2   32      0      0      0      0   
3     280000    2          2         1   27      0      0      0      0   
4      30000    2          1         2   27      0      0     -1      0   

   PAY_5  ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0      0  ...      48272      49478      51242      3028      3023      3000   
1     -1  ...       1964       1883       1538      3230      3011      1964   
2      0  ...      94856      86461      83650      1808     69563      2891   
3      0  ...     257689     193231     191143     11052      9563     15017   
4      0  ...       1814          0          0      1000       664      1500   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month  
0     

In [14]:
report = sv.analyze(df)
report.show_html("creditcard_eda_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report creditcard_eda_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Has 24000 observations and 24 columns
Data seems to be imbalanced
77.9% did not default, while the other 22.1% did

In [16]:
target_col = "default payment next month"

X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget proportion:")
print(y.value_counts(normalize=True))

X_train shape: (19200, 23)
X_test shape: (4800, 23)

Target distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

Target proportion:
default payment next month
0    0.778792
1    0.221208
Name: proportion, dtype: float64


In [17]:
rf_default = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    n_jobs=-1
)

xgb_default = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

# Random Forest
rf_default.fit(X_train, y_train)
y_pred_rf = rf_default.predict(X_test)
y_prob_rf = rf_default.predict_proba(X_test)[:, 1]

# XGBoost
xgb_default.fit(X_train, y_train)
y_pred_xgb = xgb_default.predict(X_test)
y_prob_xgb = xgb_default.predict_proba(X_test)[:, 1]

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1:", f1_score(y_test, y_pred_rf, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))

print("\nXGBoost Results")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1:", f1_score(y_test, y_pred_xgb, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob_xgb))

Random Forest Results
Accuracy: 0.8114583333333333
Precision: 0.6224648985959438
Recall: 0.3757062146892655
F1: 0.46858485026423957
ROC AUC: 0.7606088384273492

XGBoost Results
Accuracy: 0.8102083333333333
Precision: 0.6142208774583964
Recall: 0.3822975517890772
F1: 0.4712710388856645
ROC AUC: 0.7602205274077298


In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv_scores = cross_val_score(
    RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1),
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

xgb_cv_scores = cross_val_score(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Random Forest Cross-Validation Results")
print("Mean ROC AUC:", rf_cv_scores.mean())
print("Std ROC AUC:", rf_cv_scores.std())

print("\nXGBoost Cross-Validation Results")
print("Mean ROC AUC:", xgb_cv_scores.mean())
print("Std ROC AUC:", xgb_cv_scores.std())

Random Forest Cross-Validation Results
Mean ROC AUC: 0.7606719106291153
Std ROC AUC: 0.007779151614896692

XGBoost Cross-Validation Results
Mean ROC AUC: 0.7624983508477018
Std ROC AUC: 0.0071649801707831485


XGBoost is slightly better than random forest
XGbost has better overall average performance based on the ROC AUC.
However, both are similar

In [19]:
print("Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf, digits=4))

print("XGBoost Classification Report")
print(classification_report(y_test, y_pred_xgb, digits=4))

Random Forest Classification Report
              precision    recall  f1-score   support

           0     0.8406    0.9353    0.8854      3738
           1     0.6225    0.3757    0.4686      1062

    accuracy                         0.8115      4800
   macro avg     0.7315    0.6555    0.6770      4800
weighted avg     0.7923    0.8115    0.7932      4800

XGBoost Classification Report
              precision    recall  f1-score   support

           0     0.8415    0.9318    0.8843      3738
           1     0.6142    0.3823    0.4713      1062

    accuracy                         0.8102      4800
   macro avg     0.7279    0.6570    0.6778      4800
weighted avg     0.7912    0.8102    0.7930      4800



XGboost appears to be better at identifying positive defaults due ot the higher recall and F1-Score

In [20]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

rf_balanced = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    class_weight="balanced",
    n_jobs=-1
)

xgb_balanced = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

rf_balanced_scores = cross_val_score(
    rf_balanced,
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

xgb_balanced_scores = cross_val_score(
    xgb_balanced,
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Balanced Random Forest Results")
print("Mean ROC AUC:", rf_balanced_scores.mean())
print("Std ROC AUC:", rf_balanced_scores.std())

print("\nBalanced XGBoost Results")
print("Mean ROC AUC:", xgb_balanced_scores.mean())
print("Std ROC AUC:", xgb_balanced_scores.std())

Balanced Random Forest Results
Mean ROC AUC: 0.7615235257019858
Std ROC AUC: 0.007268240381769727

Balanced XGBoost Results
Mean ROC AUC: 0.7584877766785788
Std ROC AUC: 0.0066827922986005435


The ROC AUC for random forest improved slightly/standard deviation decreased. The performance of XGBoost did decrease slightly

In [21]:
xgb_subsample = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    subsample=0.8
)

xgb_subsample_scores = cross_val_score(
    xgb_subsample,
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("XGBoost with Subsample = 0.8 Results")
print("Mean ROC AUC:", xgb_subsample_scores.mean())
print("Std ROC AUC:", xgb_subsample_scores.std())

XGBoost with Subsample = 0.8 Results
Mean ROC AUC: 0.7524717837827358
Std ROC AUC: 0.007415309964235046


XGBoost with subsample = 0.8 decrease slightly. The ROC AUC lowered. Adding subsample did not help the performance

In [22]:
rf_tuned = RandomForestClassifier(
    random_state=42,
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    class_weight="balanced",
    n_jobs=-1
)

xgb_tuned = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8
)

rf_tuned_scores = cross_val_score(
    rf_tuned,
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

xgb_tuned_scores = cross_val_score(
    xgb_tuned,
    X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Tuned Random Forest Results")
print("Mean ROC AUC:", rf_tuned_scores.mean())
print("Std ROC AUC:", rf_tuned_scores.std())

print("\nTuned XGBoost Results")
print("Mean ROC AUC:", xgb_tuned_scores.mean())
print("Std ROC AUC:", xgb_tuned_scores.std())

Tuned Random Forest Results
Mean ROC AUC: 0.7750977926518384
Std ROC AUC: 0.007282797049626134

Tuned XGBoost Results
Mean ROC AUC: 0.77883569705271
Std ROC AUC: 0.008970714983245436


In [24]:
rf_tuned.fit(X_train, y_train)
y_pred_rf_tuned = rf_tuned.predict(X_test)

xgb_tuned.fit(X_train, y_train)
y_pred_xgb_tuned = xgb_tuned.predict(X_test)

print("Tuned Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf_tuned, digits=4))

print("Tuned XGBoost Classification Report")
print(classification_report(y_test, y_pred_xgb_tuned, digits=4))

Tuned Random Forest Classification Report
              precision    recall  f1-score   support

           0     0.8681    0.8596    0.8638      3738
           1     0.5223    0.5405    0.5312      1062

    accuracy                         0.7890      4800
   macro avg     0.6952    0.7000    0.6975      4800
weighted avg     0.7916    0.7890    0.7902      4800

Tuned XGBoost Classification Report
              precision    recall  f1-score   support

           0     0.8838    0.7953    0.8372      3738
           1     0.4673    0.6318    0.5372      1062

    accuracy                         0.7592      4800
   macro avg     0.6755    0.7136    0.6872      4800
weighted avg     0.7916    0.7592    0.7709      4800



The tuned RF had a balance of precision and recall, makking it better at identifying defaults cases without losing too much precision. 

The tuned XGBoost showed a better ability to identify a larger amount of default cases

In [25]:
results_df = pd.DataFrame({
    "Model": [
        "Random Forest Default",
        "XGBoost Default",
        "Random Forest Balanced",
        "XGBoost Balanced",
        "XGBoost Subsample 0.8",
        "Random Forest Tuned",
        "XGBoost Tuned"
    ],
    "Mean ROC AUC": [
        rf_cv_scores.mean(),
        xgb_cv_scores.mean(),
        rf_balanced_scores.mean(),
        xgb_balanced_scores.mean(),
        xgb_subsample_scores.mean(),
        rf_tuned_scores.mean(),
        xgb_tuned_scores.mean()
    ],
    "Std ROC AUC": [
        rf_cv_scores.std(),
        xgb_cv_scores.std(),
        rf_balanced_scores.std(),
        xgb_balanced_scores.std(),
        xgb_subsample_scores.std(),
        rf_tuned_scores.std(),
        xgb_tuned_scores.std()
    ]
})

print(results_df)

                    Model  Mean ROC AUC  Std ROC AUC
0   Random Forest Default      0.760672     0.007779
1         XGBoost Default      0.762498     0.007165
2  Random Forest Balanced      0.761524     0.007268
3        XGBoost Balanced      0.758488     0.006683
4   XGBoost Subsample 0.8      0.752472     0.007415
5     Random Forest Tuned      0.775098     0.007283
6           XGBoost Tuned      0.778836     0.008971


The best out of the box model seems to be the XGBoost.